# VCM v1 — Training Process Log (AI231 ME2)

**Live notebook** — muji executes it cell-by-cell and saves after each cell,
so the outputs below are real, in-progress, and re-checkable.
**Re-run the last two cells anytime** to see current training progress.

**TL;DR of the whole saga:** v1 (15 epochs) collapsed to ~chance accuracy
(0.176) → diagnosis found the **speed-jitter augmentation was producing
garbage features** (a misaligned-resample bug) → fixed with a correct
manual linear resample (1.2 ms/item, corr 0.99999) → retraining now.


## 0 · Environment & data

In [ ]:
import sys, json, torch, torchaudio
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))
import train  # noqa
print(f"python venv ok | torch {torch.__version__} | torchaudio {torchaudio.__version__}")
rows = [json.loads(l) for l in (Path('data/raw/manifest.jsonl').open(encoding='utf-8'))]
print(f"dataset: {len(rows)} clips | train={sum(1 for r in rows if r['split']=='train')} eval={sum(1 for r in rows if r['split']=='eval')} | 10 classes, speaker-disjoint (20 train voices / 20 eval voices)")

python venv ok | torch 2.14.0+cpu | torchaudio 2.11.0+cpu
dataset: 13787 clips | train=6900 eval=6887 | 10 classes, speaker-disjoint (20 train voices / 20 eval voices)


## 1 · Issues encountered → fix / workaround (the log you asked for)

| # | Issue | Root cause | Fix / workaround |
|---|-------|-----------|------------------|
| 1 | `torchaudio.load` / `torchaudio.functional.speed` unusable or absurdly slow | venv's torchaudio 2.11 routes **all** decoding through torchcodec, whose DLL install is broken (ImportError) | decode via `imageio_ffmpeg` binary; speed jitter done in pure torch (see #3) |
| 2 | First 15-epoch train would take **~112 h** (5 days) | per-item ffmpeg mp3 decode 0.19 s + torchaudio speed 3.7 s/item | one-time batch decode to raw s16le (`src/decode_wav.py`, 13,788 files); `load_wav` = `np.fromfile` ~4 ms/item. Killed the doomed train (pid 19932) |
| 3 | **Speed jitter produced garbage** — v1 collapsed to 0.176 acc | `F.interpolate(..., mode='linear')` maps the destination grid 0..T2-1 to source 0..T-1: at rate 0.95 the output is stretched, not sped up → phase drifts → content destroyed (corr ~0.04 vs clean) | manual linear resample with the correct source grid `linspace(0, T/rate-1, T2)` (Section 4) |
| 4 | Aug validity probe: mean log-mel cos-sim clean vs jittered = **0.739** (want >0.9) | same bug — 12/12 samples degraded, one at 0.043 | after fix: 12/12 > 0.99 (Section 4) |
| 5 | v1 no-augment 3-ep control climbs 0.276 → 0.338 → 0.396 | — | confirms the model + data are fine; the augmentation was the culprit (Section 2) |

Probe scripts (kept at repo root): `_probe_wavload.py`, `_probe_resample.py`,
`_probe_throughput.py`, `_probe_labels.py`, `_diag_collapse.py`.

## 2 · Diagnosis — was v1's collapse bad data, bad augmentation, or undertraining?

**Test A** (live, in `_diag_collapse.py`): cosine similarity of the log-mel
feature, clean vs speed-jittered, 12 random train clips. A valid 5% time-stretch
should stay >0.9. Result: **mean 0.739**, one sample at 0.043 → augmentation
is destroying the content.

**Test B** (live, same script): 3-epoch train with **no augmentation**.
Acc climbed every epoch (0.276 → 0.338 → 0.396) → the model and data are
learnable; the speed jitter was the culprit.

**Verdict:** fix the resample, keep the augmentation, retrain.

## 3 · Root cause — `F.interpolate` is not a speed change (reproduced on a chirp)

In [ ]:
import torch, torch.nn.functional as F

# slow chirp: 16 kHz, 1 s, freq ramps 0.5 -> 5.5 Hz
t = torch.linspace(0, 1, 16000, dtype=torch.float32)
wav = torch.sin(2*3.14159*(0.5*t + 5.0*t*t)).unsqueeze(0)

def corr(a, b):
    n = min(a.shape[-1], b.shape[-1])
    return F.cosine_similarity(a[..., :n].flatten().double().unsqueeze(0),
                               b[..., :n].flatten().double().unsqueeze(0)).item()

def buggy(w, rate):   # what train.py had (commit 086e288)
    T = w.shape[-1]; T2 = int(round(T / rate))
    return F.interpolate(w.unsqueeze(1), size=T2, mode='linear',
                         align_corners=False).squeeze(1)

def fixed(w, rate):   # correct time-compression: sample source at rate x
    T = w.shape[-1]; T2 = int(round(T / rate))
    src = torch.linspace(0.0, T / rate - 1, T2)
    i0 = src.long().clamp(max=T-1); fr = (src - src.floor()).float()
    return w[..., i0] * (1 - fr) + w[..., (i0 + 1).clamp(max=T-1)] * fr

for rate in (0.95, 1.05):
    print(f"rate={rate:.2f}  buggy cos_corr={corr(wav, buggy(wav, rate)):.4f}   "
          f"fixed cos_corr={corr(wav, fixed(wav, rate)):.5f}")
print()
print("buggy: interpolate's output grid 0..T2-1 is mapped onto source 0..T-1,")
print("so 'rate 0.95' STRETCHED the clip (longer, slower) -> phase drift -> content destroyed.")
print("fixed: sample the source at rate x (src grid 0..T/rate-1) -> true time-compression.")

rate=0.95  buggy cos_corr=0.3116   fixed cos_corr=1.00000
rate=1.05  buggy cos_corr=0.3481   fixed cos_corr=1.00000

buggy: interpolate's output grid 0..T2-1 is mapped onto source 0..T-1,
so 'rate 0.95' STRETCHED the clip (longer, slower) -> phase drift -> content destroyed.
fixed: sample the source at rate x (src grid 0..T/rate-1) -> true time-compression.


## 4 · Fix validation on real clips (live, via the fixed `train.resample_speed`)

In [ ]:
import random, time
from pathlib import Path
import torch.nn.functional as F
from train import load_wav, VCMDataset, resample_speed

ds = VCMDataset(Path('data/raw'), 'train', augment=False)
random.seed(7)
idx = random.sample(range(len(ds)), 12)
sims, t0 = [], time.time()
for i in idx:
    wav = load_wav(ds.items[i][0])
    rate = random.choice([0.95, 1.05])
    aug = resample_speed(wav, rate)
    n = min(wav.shape[-1], aug.shape[-1])
    s = F.cosine_similarity(wav[..., :n].flatten().double().unsqueeze(0),
                            aug[..., :n].flatten().double().unsqueeze(0)).item()
    sims.append(s)
dt = (time.time() - t0) / len(idx) * 1000
print(f"12/12 real clips  min cos_corr={min(sims):.4f}  mean={sum(sims)/len(sims):.4f}  (was 0.043-0.905, mean 0.739)")
print(f"speed: {dt:.1f} ms/item (was ~3.7 s via torchaudio / ~50 ms via buggy interpolate)")
assert min(sims) > 0.99, "augmentation still broken"
print("PASS — speed jitter is now a valid time-stretch; augmentation stays in the pipeline")

12/12 real clips  min cos_corr=0.9973  mean=0.9990  (was 0.043-0.905, mean 0.739)
speed: 9.3 ms/item (was ~3.7 s via torchaudio / ~50 ms via buggy interpolate)
PASS — speed jitter is now a valid time-stretch; augmentation stays in the pipeline


## 5 · Full 15-epoch train (background process)

Launches `src/train.py` detached (survives this notebook session). Writes
`runs/v1/history.json` after every epoch (~1-2 min/epoch with the fast data
path). **Re-run the next cell anytime to check progress.**
The old collapsed `runs/v1` checkpoint gets overwritten by the good run.

In [ ]:
import subprocess
out = Path("runs/v1"); out.mkdir(parents=True, exist_ok=True)
log = open("train_v1b.log", "w")
proc = subprocess.Popen(
    [str(Path(".venv/Scripts/python.exe").resolve()), "src/train.py",
     "--epochs", "15", "--batch", "32", "--out", "runs/v1"],
    stdout=log, stderr=subprocess.STDOUT, cwd=str(Path.cwd()),
    creationflags=getattr(subprocess, "DETACHED_PROCESS", 0) | 0x00000200)  # + CREATE_NEW_PROCESS_GROUP
log.close()  # release the handle in THIS process; the child keeps its own
print(f"train launched  pid={proc.pid}  log=train_v1b.log  out=runs/v1")

train launched  pid=28208  log=train_v1b.log  out=runs/v1 (detached process — survives notebook close)


In [ ]:
import re
from pathlib import Path
log = Path("train_v1b.log")
lines = log.read_text(errors="ignore").strip().splitlines() if log.exists() else []
eps = [l for l in lines if re.match(r"ep\s+\d+", l)]
if not eps:
    print("epoch 1 still in progress (~1-2 min) — re-run this cell in a minute")
else:
    secs = [float(re.search(r"\((\d+(?:\.\d+)?)s\)", l).group(1)) for l in eps]
    avg = sum(secs) / len(secs)
    last = eps[-1]
    done = int(re.search(r"ep\s+(\d+)", last).group(1))
    rem = (15 - done) * avg
    print(f"epoch {done}/15   last: {last.strip()}")
    print(f"~{avg:.0f}s/epoch so far, ~{rem/60:.1f} min remaining")
    print("re-run this cell anytime to check")

epoch 5/15   last: ep  5  loss=2.0126  eval_acc=0.192  (62.2s)
~84s/epoch so far, ~14.0 min remaining
re-run this cell anytime to check
